# Task A Notebook v0 — On-Demand User Profiles + Rating/Review Simulation

Goal for today:
1. Load the prepared train/internal validation splits.
2. Build a compact user evidence packet from train history only.
3. Generate a user profile on demand using Gemini or DeepSeek.
4. Cache the profile.
5. Predict rating + generate review for a held-out item.

This notebook is intentionally simple and iteration-friendly. Once the prompts/methods are stable, we can move the code into clean scripts for the app/container.

In [1]:
# Optional, only run if needed:
# %pip install pandas pyarrow requests tqdm

import os
import json
import math
import time
import hashlib
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)

In [3]:
# -------------------------
# Config
# -------------------------

OUTPUT_DIR = Path("data/processed/baseline_v0")
SPLITS_DIR = OUTPUT_DIR / "splits"
CACHE_DIR = OUTPUT_DIR / "cache"
PROFILE_CACHE_DIR = CACHE_DIR / "user_profiles"
PROFILE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = SPLITS_DIR / "train.parquet"
INTERNAL_VAL_PATH = SPLITS_DIR / "internal_val.parquet"
ITEMS_PATH = OUTPUT_DIR / "items.parquet"  # exists only if you did not run --skip-meta

# Switch this as needed.
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "gemini")  # "gemini" or "deepseek"
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash-lite")
DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash")

# Keep low while iterating.
TEMPERATURE = 0.2
MAX_OUTPUT_TOKENS = 1800

print("Provider:", LLM_PROVIDER)
print("Train exists:", TRAIN_PATH.exists())
print("Internal val exists:", INTERNAL_VAL_PATH.exists())
print("Items exists:", ITEMS_PATH.exists())

Provider: gemini
Train exists: True
Internal val exists: True
Items exists: True


In [4]:
# -------------------------
# Load data
# -------------------------

train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(INTERNAL_VAL_PATH)

if ITEMS_PATH.exists():
    items_df = pd.read_parquet(ITEMS_PATH)
else:
    # Fallback: minimal item table from interactions only.
    items_df = (
        pd.concat([train_df[["domain", "parent_asin"]], val_df[["domain", "parent_asin"]]], ignore_index=True)
        .drop_duplicates()
        .assign(title="", categories="", description="", features="", store="", price=None)
    )

# Make lookup fast.
item_lookup = {
    (str(r.domain), str(r.parent_asin)): r._asdict()
    for r in items_df.fillna("").itertuples(index=False)
}

print("train:", train_df.shape)
print("internal_val:", val_df.shape)
print("items:", items_df.shape)

train_df.head(2)

train: (876848, 13)
internal_val: (106115, 13)
items: (524430, 9)


,domain,user_id,asin,parent_asin,rating,review_title,review_text,timestamp,verified_purchase,helpful_vote,rn,n_user_reviews,split
0,Books,AE23HUJD2RENUFCMHPVVC3F64KRQ,1603863028,1603863028,5.0,Important 19th-century Russian novel,"This novel is probably richer for those who know a fair amount about Russian history, and particularly Russian intellectual movements, during the 19th centu...",1374931805000,True,0,1,30,train
1,Books,AE23HUJD2RENUFCMHPVVC3F64KRQ,0803737599,0803737599,5.0,Grandchildren love it!,"We've been having a wonderful time reading this very imaginative, often hilarious book. I was curious about this new work by Jennifer Allison, because my ol...",1379427570000,True,0,2,30,train


In [ ]:
# from pathlib import Path
# import pandas as pd
# import numpy as np

# # -----------------------------
# # Config
# # -----------------------------
# PROCESSED_DIR = Path("data/processed/baseline_v0")

# SPLITS_DIR = PROCESSED_DIR / "splits"
# ITEMS_PATH = PROCESSED_DIR / "items.parquet"

# # For fast Task A prompt iteration.
# # Increase/decrease depending on how many examples survive.
# MIN_TRAIN_REVIEWS_PER_USER = 12
# MIN_INTERNAL_VAL_REVIEWS_PER_USER = 1
# MIN_ITEM_TRAIN_REVIEWS = 3

# REQUIRE_ITEM_TITLE = True
# REQUIRE_ITEM_METADATA = True

# # -----------------------------
# # Load data
# # -----------------------------
# train_df = pd.read_parquet(SPLITS_DIR / "train.parquet")
# internal_val_df = pd.read_parquet(SPLITS_DIR / "internal_val.parquet")
# shadow_test_df = pd.read_parquet(SPLITS_DIR / "shadow_test.parquet")

# items_df = pd.read_parquet(ITEMS_PATH)

# print("Raw loaded:")
# print("train:", train_df.shape)
# print("internal_val:", internal_val_df.shape)
# print("shadow_test:", shadow_test_df.shape)
# print("items:", items_df.shape)

# # -----------------------------
# # Clean item metadata
# # -----------------------------
# items_df = items_df.copy()

# for col in ["title", "main_category", "categories", "features", "description", "store"]:
#     if col not in items_df.columns:
#         items_df[col] = ""
#     items_df[col] = items_df[col].fillna("").astype(str).str.strip()

# items_df = items_df.drop_duplicates(["domain", "parent_asin"])

# def has_useful_metadata(row):
#     title_ok = bool(row["title"]) if REQUIRE_ITEM_TITLE else True

#     metadata_text = " ".join([
#         row.get("main_category", ""),
#         row.get("categories", ""),
#         row.get("features", ""),
#         row.get("description", ""),
#         row.get("store", ""),
#     ]).strip()

#     metadata_ok = bool(metadata_text) if REQUIRE_ITEM_METADATA else True

#     return title_ok and metadata_ok

# items_df["has_useful_metadata"] = items_df.apply(has_useful_metadata, axis=1)

# valid_meta_items = items_df.loc[
#     items_df["has_useful_metadata"],
#     ["domain", "parent_asin"]
# ].drop_duplicates()

# # -----------------------------
# # Keep only items that:
# # 1. have valid metadata
# # 2. appear enough times in train
# # -----------------------------
# item_train_counts = (
#     train_df
#     .groupby(["domain", "parent_asin"])
#     .size()
#     .reset_index(name="n_train_reviews_for_item")
# )

# viable_items = (
#     valid_meta_items
#     .merge(item_train_counts, on=["domain", "parent_asin"], how="inner")
# )

# viable_items = viable_items[
#     viable_items["n_train_reviews_for_item"] >= MIN_ITEM_TRAIN_REVIEWS
# ].copy()

# print("\nViable items:", viable_items.shape)

# # -----------------------------
# # Filter splits to viable items
# # -----------------------------
# def filter_to_viable_items(df, viable_items):
#     return (
#         df
#         .merge(
#             viable_items[["domain", "parent_asin"]],
#             on=["domain", "parent_asin"],
#             how="inner"
#         )
#         .copy()
#     )

# train_df = filter_to_viable_items(train_df, viable_items)
# internal_val_df = filter_to_viable_items(internal_val_df, viable_items)
# shadow_test_df = filter_to_viable_items(shadow_test_df, viable_items)

# # -----------------------------
# # Keep only users with enough usable history
# # and at least one usable validation target
# # -----------------------------
# train_user_counts = (
#     train_df
#     .groupby("user_id")
#     .size()
#     .reset_index(name="n_train_reviews")
# )

# val_user_counts = (
#     internal_val_df
#     .groupby("user_id")
#     .size()
#     .reset_index(name="n_internal_val_reviews")
# )

# viable_users = (
#     train_user_counts
#     .merge(val_user_counts, on="user_id", how="inner")
# )

# viable_users = viable_users[
#     (viable_users["n_train_reviews"] >= MIN_TRAIN_REVIEWS_PER_USER)
#     & (viable_users["n_internal_val_reviews"] >= MIN_INTERNAL_VAL_REVIEWS_PER_USER)
# ].copy()

# print("Viable users:", viable_users.shape)

# valid_user_ids = set(viable_users["user_id"])

# train_df = train_df[train_df["user_id"].isin(valid_user_ids)].copy()
# internal_val_df = internal_val_df[internal_val_df["user_id"].isin(valid_user_ids)].copy()
# shadow_test_df = shadow_test_df[shadow_test_df["user_id"].isin(valid_user_ids)].copy()

# # -----------------------------
# # Attach metadata to filtered item table
# # -----------------------------
# items_df = (
#     items_df
#     .merge(
#         viable_items[["domain", "parent_asin", "n_train_reviews_for_item"]],
#         on=["domain", "parent_asin"],
#         how="inner"
#     )
#     .copy()
# )

# # Useful lookup for Task A target item metadata.
# item_lookup = {
#     (row["domain"], row["parent_asin"]): row.to_dict()
#     for _, row in items_df.iterrows()
# }

# # -----------------------------
# # Sort chronologically after filtering
# # -----------------------------
# sort_cols = ["user_id", "timestamp", "domain", "parent_asin"]

# train_df = train_df.sort_values(sort_cols).reset_index(drop=True)
# internal_val_df = internal_val_df.sort_values(sort_cols).reset_index(drop=True)
# shadow_test_df = shadow_test_df.sort_values(sort_cols).reset_index(drop=True)

# # -----------------------------
# # Quick summaries
# # -----------------------------
# print("\nFiltered loaded:")
# print("train:", train_df.shape)
# print("internal_val:", internal_val_df.shape)
# print("shadow_test:", shadow_test_df.shape)
# print("items:", items_df.shape)

# print("\nUsers:")
# print("train users:", train_df["user_id"].nunique())
# print("internal_val users:", internal_val_df["user_id"].nunique())
# print("shadow_test users:", shadow_test_df["user_id"].nunique())

# print("\nDomains:")
# print("train:")
# print(train_df["domain"].value_counts())
# print("\ninternal_val:")
# print(internal_val_df["domain"].value_counts())

# print("\nTrain reviews per viable user:")
# display(
#     train_df
#     .groupby("user_id")
#     .size()
#     .describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
# )

# print("\nInternal validation reviews per viable user:")
# display(
#     internal_val_df
#     .groupby("user_id")
#     .size()
#     .describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
# )

# # -----------------------------
# # Small Task A iteration sample
# # -----------------------------
# task_a_sample = (
#     internal_val_df
#     .sample(
#         n=min(50, len(internal_val_df)),
#         random_state=42
#     )
#     .reset_index(drop=True)
# )

# print("\nTask A sample:", task_a_sample.shape)

# task_a_sample.head()

Raw loaded:
train: (876848, 13)
internal_val: (106115, 13)
shadow_test: (106115, 13)
items: (524430, 9)

Viable items: (50927, 3)
Viable users: (5978, 3)

Filtered loaded:
train: (190457, 13)
internal_val: (20408, 13)
shadow_test: (17348, 13)
items: (50927, 11)

Users:
train users: 5978
internal_val users: 5978
shadow_test users: 4975

Domains:
train:
Movies_and_TV    104696
Books             85761
Name: domain, dtype: int64

internal_val:
Movies_and_TV    11570
Books             8838
Name: domain, dtype: int64

Train reviews per viable user:


count    5978.000000
mean       31.859652
std        42.371433
min        12.000000
50%        20.000000
75%        32.000000
90%        58.000000
95%        88.150000
99%       199.000000
max       951.000000
dtype: float64


Internal validation reviews per viable user:


count    5978.000000
mean        3.413851
std         4.660538
min         1.000000
50%         2.000000
75%         4.000000
90%         7.000000
95%        10.000000
99%        22.000000
max        91.000000
dtype: float64


Task A sample: (50, 13)


,domain,user_id,asin,parent_asin,rating,review_title,review_text,timestamp,verified_purchase,helpful_vote,rn,n_user_reviews,split
0,Books,AGWKP5DUT57IH4JZMNJ2CD6R25XA,B01FQLZS1S,B01FQLZS1S,4.0,Twists and turns,This is the 4th book in the John Puller series. It is an entertaining mystery with lots of twists and turns and with engaging characters. I enjoyed the book.,1620323170803,False,0,67,88,internal_val
1,Movies_and_TV,AFDA7KYL6PM63BUDXHZ73NQCLWBQ,B000J10EQU,B000J10EQU,5.0,love the movie,ty,1563206623026,True,0,47,62,internal_val
2,Books,AFI3RYFT4J4DQWQU3CFDIY5SZT6A,B01HMNETJA,B01HMNETJA,5.0,Orphan X and Nowhere Man are excellent,"Orphan X was so good I went right on to Nowhere Man. Fast-paced and exciting, these books are the perfect combination of thriller, technology, and humanity.",1489889356000,True,0,39,49,internal_val
3,Books,AECH5QWPTTIRDEFDCWB67TREHIIQ,B01KI00WV8,B01KI00WV8,5.0,The Tracker,Great book kept reader interested in it to the very end. Will look for more from this author in the future.p,1599643836148,True,0,31,39,internal_val
4,Movies_and_TV,AHOAKTHD5FA7MWLV4I2IBOYZCCAA,B00ZJ2GOZO,B00ZJ2GOZO,5.0,This is amazing! My discs seem to work,"This is amazing! My discs seem to work. I love them all. of course, the last movie she could have at least broken a high heel! The 1st and the 3rd will alwa...",1484361002000,True,0,73,102,internal_val


In [26]:
from pathlib import Path
import json

# -----------------------------
# Save filtered Task A iteration dataset
# -----------------------------
TASK_A_DIR = Path("data/processed/task_a_iteration_v0")
TASK_A_DIR.mkdir(parents=True, exist_ok=True)

train_out = TASK_A_DIR / "train.parquet"
internal_val_out = TASK_A_DIR / "internal_val.parquet"
shadow_test_out = TASK_A_DIR / "shadow_test.parquet"
items_out = TASK_A_DIR / "items.parquet"
summary_out = TASK_A_DIR / "summary.json"

train_df.to_parquet(train_out, index=False)
internal_val_df.to_parquet(internal_val_out, index=False)
shadow_test_df.to_parquet(shadow_test_out, index=False)
items_df.to_parquet(items_out, index=False)

summary = {
    "train_shape": list(train_df.shape),
    "internal_val_shape": list(internal_val_df.shape),
    "shadow_test_shape": list(shadow_test_df.shape),
    "items_shape": list(items_df.shape),

    "train_users": int(train_df["user_id"].nunique()),
    "internal_val_users": int(internal_val_df["user_id"].nunique()),
    "shadow_test_users": int(shadow_test_df["user_id"].nunique()),

    "train_domain_counts": train_df["domain"].value_counts().to_dict(),
    "internal_val_domain_counts": internal_val_df["domain"].value_counts().to_dict(),
    "shadow_test_domain_counts": shadow_test_df["domain"].value_counts().to_dict(),

    "n_items": int(items_df[["domain", "parent_asin"]].drop_duplicates().shape[0]),
}

with open(summary_out, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved filtered Task A iteration files:")
print("train:", train_out)
print("internal_val:", internal_val_out)
print("shadow_test:", shadow_test_out)
print("items:", items_out)
print("summary:", summary_out)

Saved filtered Task A iteration files:
train: data/processed/task_a_iteration_v0/train.parquet
internal_val: data/processed/task_a_iteration_v0/internal_val.parquet
shadow_test: data/processed/task_a_iteration_v0/shadow_test.parquet
items: data/processed/task_a_iteration_v0/items.parquet
summary: data/processed/task_a_iteration_v0/summary.json


In [28]:
from pathlib import Path
import pandas as pd
import json

TASK_A_DIR = Path("data/processed/task_a_iteration_v0")

train_df = pd.read_parquet(TASK_A_DIR / "train.parquet")
internal_val_df = pd.read_parquet(TASK_A_DIR / "internal_val.parquet")
shadow_test_df = pd.read_parquet(TASK_A_DIR / "shadow_test.parquet")
items_df = pd.read_parquet(TASK_A_DIR / "items.parquet")

with open(TASK_A_DIR / "summary.json", "r", encoding="utf-8") as f:
    saved_summary = json.load(f)

item_lookup = {
    (row["domain"], row["parent_asin"]): row.to_dict()
    for _, row in items_df.iterrows()
}

print("Loaded filtered Task A iteration data:")
print("train:", train_df.shape)
print("internal_val:", internal_val_df.shape)
print("shadow_test:", shadow_test_df.shape)
print("items:", items_df.shape)

saved_summary

Loaded filtered Task A iteration data:
train: (190457, 13)
internal_val: (20408, 13)
shadow_test: (17348, 13)
items: (50927, 11)


{'train_shape': [190457, 13],
 'internal_val_shape': [20408, 13],
 'shadow_test_shape': [17348, 13],
 'items_shape': [50927, 11],
 'train_users': 5978,
 'internal_val_users': 5978,
 'shadow_test_users': 4975,
 'train_domain_counts': {'Movies_and_TV': 104696, 'Books': 85761},
 'internal_val_domain_counts': {'Movies_and_TV': 11570, 'Books': 8838},
 'shadow_test_domain_counts': {'Movies_and_TV': 9848, 'Books': 7500},
 'n_items': 50927}

In [31]:
train_df[train_df['user_id'] == 'AE225Z2VRWT6GPTOMA4H4O3H2KVQ']

,domain,user_id,asin,parent_asin,rating,review_title,review_text,timestamp,verified_purchase,helpful_vote,rn,n_user_reviews,split
0,Movies_and_TV,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B001GUYB08,B001GUYB08,5.0,Five Stars,Love ALL the Resident Evil movies! tHANK YOU aMAZON!,1269056892000,True,0,3,40,train
1,Movies_and_TV,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B000YPUF9W,B000YPUF9W,5.0,Resident Evil: Extinction,"What can I say? The price was right. My DVD arrived in a timely manner, and in good shape. No cracked jewel case like sometimes happens to DVD's or CD's. I'...",1363045606000,True,0,7,40,train
2,Movies_and_TV,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B002ZG98UA,B002ZG98UA,5.0,Resident Evil: Afterlife,I loved ALL of the Resident Evil movies! Milla Jovovich is Awesome! I remember my friends playing this when is was just a play-station game..even I got hoo...,1363045908000,True,0,8,40,train
3,Movies_and_TV,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B009MO5960,B009MO5960,3.0,Blue - ray,"I wanted to add this to my collection of Resident Evil's, unfortunately, I wasn't paying attention, and I ordered the Blue-Ray, which I cannot play on anyth...",1363046080000,True,0,10,40,train
4,Books,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B004D4ZQ3K,B004D4ZQ3K,5.0,"Valley if Death, Zombie Trailer Park","WHY isn't this a movie!? Excellent story, and I love the way William Bebb writes! I got it for my Kindle. THANK YOU!!",1363046495000,True,1,11,40,train
5,Books,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B000UDNBRQ,B000UDNBRQ,5.0,Koontz is always a winner!,"I ordered this book for my Kindle. Koontz and King, two of my favorite authors. Koontz keeps you on the edge of your seat, as always. He knows how to weave ...",1376276267000,True,0,12,40,train
6,Books,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B000SEGJ5S,B000SEGJ5S,5.0,Love my Dictionary!,"There are many times, while reading my Kindle book, that I need to look up a word, and vola! There it is, in my Kindle, ready to show me a meaning of a word...",1376276699000,True,0,13,40,train
7,Books,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B000UZJREU,B000UZJREU,5.0,Duma Key: A Novel,"Absolutely loved this book! Stephen King just keeps going, and going, and going.... Could not put it down. Didn't want to put it down! I absolutely recom...",1377994462000,True,0,14,40,train
8,Movies_and_TV,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B00AFDN5Y0,B00AFDN5Y0,5.0,A Perfect Ending,"I really liked this movie. Good story line and interesting characters. Fun and sensual.What more could you ask for? that's why bought it, to share with fri...",1377995601000,True,2,16,40,train
9,Books,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B008ED5G9A,B008ED5G9A,5.0,Jack reacher,"This was my first time reading this type of book. And I loved it! I could imagine it being turned into a movie...oh! I think it finally was, with Tom Cruis...",1384379083000,True,0,18,40,train


In [30]:
internal_val_df[internal_val_df['user_id'] == 'AE225Z2VRWT6GPTOMA4H4O3H2KVQ']

,domain,user_id,asin,parent_asin,rating,review_title,review_text,timestamp,verified_purchase,helpful_vote,rn,n_user_reviews,split
0,Movies_and_TV,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B0000E2R6P,B0000E2R6P,5.0,EXCELLENT old fashioned Horror,Yea Mr. Silva! EXCELLENT old fashioned Horror!!,1415579707000,True,0,31,40,internal_val
1,Books,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B005OH9A6E,B005OH9A6E,5.0,cool movie!,I haven't finished reading it yet...almost done...I am so gone! Taken away and right in there wit the characters! Dean Koontz has done it again...would make...,1416969499000,True,1,32,40,internal_val


In [33]:
shadow_test_df[shadow_test_df['user_id'] == 'AE225Z2VRWT6GPTOMA4H4O3H2KVQ']

,domain,user_id,asin,parent_asin,rating,review_title,review_text,timestamp,verified_purchase,helpful_vote,rn,n_user_reviews,split
0,Books,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B00GEEB52S,B00GEEB52S,5.0,And it looks good! Am finishing another book,"I only read a few pates of this book...preview. And it looks good! Am finishing another book, and when I finish, I'm jumping into this with both feet! Mr....",1416969653000,True,0,33,40,shadow_test
1,Books,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B00FPR89BA,B00FPR89BA,5.0,Five Stars,Watch out...Koontz will get 'cha! But you'll love get being 'gotten' by Koontz!,1421025494000,True,0,35,40,shadow_test


In [32]:
items_df[items_df['parent_asin'] == 'B0000E2R6P']

,domain,parent_asin,main_category,title,store,categories,features,description,price,has_useful_metadata,n_train_reviews_for_item
36671,Movies_and_TV,B0000E2R6P,Movies & TV,Jeepers Creepers 2,"Jonathan Breck (Actor), Ray Wise (Actor), Victor Salva (Director, Writer) & 0 more Rated: R Format:...",Movies & TV MGM Home Entertainment All MGM Titles,,"Product Description SET A FEW DAYS AFTER THE ORIGINAL, A CHAMPIONSHIP BASKETBALL TEAM'S BUS IS ATTACKED BY THE CREEPER, THE WINGED, FLESH-EATING TERROR, ON ...",8.56,True,4


In [5]:
# -------------------------
# Small helpers
# -------------------------

def clean_text(x: Any, max_chars: Optional[int] = None) -> str:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        s = ""
    elif isinstance(x, list):
        s = " ".join(clean_text(v) for v in x)
    elif isinstance(x, dict):
        s = json.dumps(x, ensure_ascii=False)
    else:
        s = str(x)
    s = re.sub(r"\s+", " ", s).strip()
    if max_chars and len(s) > max_chars:
        return s[:max_chars].rsplit(" ", 1)[0] + "..."
    return s

def safe_json_loads(text: str) -> Dict[str, Any]:
    """Parse JSON even when the model wraps it in markdown or extra text."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(text[start:end+1])
        raise

def rating_distribution(series: pd.Series) -> Dict[str, int]:
    counts = series.round().astype(int).value_counts().to_dict()
    return {str(k): int(counts.get(k, 0)) for k in range(1, 6)}

def summarize_item(domain: str, parent_asin: str, max_chars: int = 700) -> Dict[str, Any]:
    meta = item_lookup.get((str(domain), str(parent_asin)), {})
    return {
        "domain": str(domain),
        "parent_asin": str(parent_asin),
        "title": clean_text(meta.get("title", ""), 180),
        "store": clean_text(meta.get("store", ""), 80),
        "categories": clean_text(meta.get("categories", ""), 250),
        "features": clean_text(meta.get("features", ""), max_chars),
        "description": clean_text(meta.get("description", ""), max_chars),
        "price": meta.get("price", None),
    }

def compact_review_row(row: pd.Series, include_item_meta: bool = True) -> Dict[str, Any]:
    item = summarize_item(row["domain"], row["parent_asin"], max_chars=300) if include_item_meta else {}
    return {
        "domain": row["domain"],
        "parent_asin": row["parent_asin"],
        "rating": float(row["rating"]),
        "review_title": clean_text(row.get("review_title", ""), 120),
        "review_text": clean_text(row.get("review_text", ""), 700),
        "timestamp": int(row["timestamp"]),
        "verified_purchase": bool(row.get("verified_purchase", False)),
        "helpful_vote": int(row.get("helpful_vote", 0)),
        "item_title": item.get("title", ""),
        "item_categories": item.get("categories", ""),
    }

In [35]:
def build_user_evidence_packet(
    user_id: str,
    target_domain: Optional[str] = None,
    target_parent_asin: Optional[str] = None,
    train: pd.DataFrame = train_df,
    max_examples: int = 12,
    n_recent: int = 4,
    n_high: int = 3,
    n_low: int = 3,
    n_same_domain: int = 4,
) -> Dict[str, Any]:
    """
    Build a compact, train-only evidence packet for one user.

    Selection strategy:
    - rating stats and domain counts
    - recent reviews
    - high-rating examples
    - low-rating examples
    - same-domain examples for the target domain
    Deduplicates examples and caps prompt size.
    """
    user_hist = train[train["user_id"].astype(str) == str(user_id)].copy()
    if user_hist.empty:
        raise ValueError(f"No train history found for user_id={user_id}")

    user_hist = user_hist.sort_values(["timestamp", "domain", "parent_asin"]).reset_index(drop=True)
    ratings = user_hist["rating"].astype(float)

    # Basic style stats.
    text_lengths = user_hist["review_text"].fillna("").map(lambda x: len(str(x).split()))
    title_lengths = user_hist["review_title"].fillna("").map(lambda x: len(str(x).split()))

    domain_counts = user_hist["domain"].value_counts().to_dict()
    domain_avg_ratings = user_hist.groupby("domain")["rating"].mean().round(3).to_dict()

    # Candidate review groups.
    recent = user_hist.tail(n_recent)
    high = user_hist.sort_values(["rating", "timestamp"], ascending=[False, False]).head(n_high)
    low = user_hist.sort_values(["rating", "timestamp"], ascending=[True, False]).head(n_low)

    groups = [("recent", recent), ("high_rating", high), ("low_rating", low)]

    if target_domain:
        same_domain = user_hist[user_hist["domain"] == target_domain].tail(n_same_domain)
        if not same_domain.empty:
            groups.append(("same_domain", same_domain))

    # Deduplicate by interaction identity.
    seen = set()
    examples = []
    for reason, df in groups:
        for _, row in df.iterrows():
            key = (row["domain"], row["parent_asin"], int(row["timestamp"]), float(row["rating"]))
            if key in seen:
                continue
            seen.add(key)
            ex = compact_review_row(row)
            ex["selection_reason"] = reason
            examples.append(ex)
            if len(examples) >= max_examples:
                break
        if len(examples) >= max_examples:
            break

    packet = {
        "user_id": str(user_id),
        "history_scope": "train_only",
        "n_train_reviews": int(len(user_hist)),
        "domains_seen": sorted([str(x) for x in user_hist["domain"].unique()]),
        "domain_counts": {str(k): int(v) for k, v in domain_counts.items()},
        "domain_avg_ratings": {str(k): float(v) for k, v in domain_avg_ratings.items()},
        "rating_stats": {
            "mean": float(ratings.mean()),
            "median": float(ratings.median()),
            "min": float(ratings.min()),
            "max": float(ratings.max()),
            "std": float(ratings.std()) if len(ratings) > 1 else 0.0,
            "distribution": rating_distribution(ratings),
        },
        "review_style_stats": {
            "avg_review_words": float(text_lengths.mean()),
            "median_review_words": float(text_lengths.median()),
            "avg_title_words": float(title_lengths.mean()),
            "empty_review_frac": float((text_lengths == 0).mean()),
            "verified_purchase_frac": float(user_hist["verified_purchase"].mean()) if "verified_purchase" in user_hist else None,
        },
        # "target_context": {
        #     "target_domain": target_domain,
        #     "target_parent_asin": target_parent_asin,
        # },
        "selected_review_examples": examples,
    }
    return packet

# Quick smoke test on one validation row.
sample_row = internal_val_df.sample(1, random_state=42).iloc[0]
packet = build_user_evidence_packet(
    user_id=sample_row["user_id"],
    # target_domain=sample_row["domain"],
    # target_parent_asin=sample_row["parent_asin"],
)
packet["n_train_reviews"], packet["rating_stats"], len(packet["selected_review_examples"])

(48,
 {'mean': 4.666666666666667,
  'median': 5.0,
  'min': 3.0,
  'max': 5.0,
  'std': 0.6302087912488811,
  'distribution': {'1': 0, '2': 0, '3': 4, '4': 8, '5': 36}},
 7)

In [36]:
# Inspect evidence packet.
print(json.dumps(packet, indent=2, ensure_ascii=False)[:5000])

{
  "user_id": "AGWKP5DUT57IH4JZMNJ2CD6R25XA",
  "history_scope": "train_only",
  "n_train_reviews": 48,
  "domains_seen": [
    "Books"
  ],
  "domain_counts": {
    "Books": 48
  },
  "domain_avg_ratings": {
    "Books": 4.667
  },
  "rating_stats": {
    "mean": 4.666666666666667,
    "median": 5.0,
    "min": 3.0,
    "max": 5.0,
    "std": 0.6302087912488811,
    "distribution": {
      "1": 0,
      "2": 0,
      "3": 4,
      "4": 8,
      "5": 36
    }
  },
  "review_style_stats": {
    "avg_review_words": 37.770833333333336,
    "median_review_words": 35.5,
    "avg_title_words": 2.375,
    "empty_review_frac": 0.0,
    "verified_purchase_frac": 0.3958333333333333
  },
  "selected_review_examples": [
    {
      "domain": "Books",
      "parent_asin": "B006KD2TP0",
      "rating": 5.0,
      "review_title": "Adventure, suspense, romance, friendship",
      "review_text": "This is another entertaining book in this series about friendship, with suspense, adventure, and romance. 

In [8]:
def call_llm(
    prompt: str,
    provider: str = LLM_PROVIDER,
    model: Optional[str] = None,
    temperature: float = TEMPERATURE,
    max_output_tokens: int = MAX_OUTPUT_TOKENS,
    json_mode: bool = True,
    timeout: int = 90,
) -> Dict[str, Any]:
    """
    Provider-switchable LLM call.

    Env vars:
      GEMINI_API_KEY for provider='gemini'
      DEEPSEEK_API_KEY for provider='deepseek'

    Returns:
      {"raw_text": "...", "json": {...} or None, "provider": "...", "model": "..."}
    """
    provider = provider.lower().strip()

    if provider == "gemini":
        api_key = os.getenv("GEMINI_API_KEY")
        if not api_key:
            raise RuntimeError("Missing GEMINI_API_KEY environment variable.")
        model = model or GEMINI_MODEL
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
        params = {"key": api_key}
        payload = {
            "contents": [{"role": "user", "parts": [{"text": prompt}]}],
            "generationConfig": {
                "temperature": temperature,
                "maxOutputTokens": max_output_tokens,
            },
        }
        if json_mode:
            payload["generationConfig"]["responseMimeType"] = "application/json"

        resp = requests.post(url, params=params, json=payload, timeout=timeout)
        resp.raise_for_status()
        data = resp.json()
        raw_text = data["candidates"][0]["content"]["parts"][0]["text"]

    elif provider == "deepseek":
        api_key = os.getenv("DEEPSEEK_API_KEY")
        if not api_key:
            raise RuntimeError("Missing DEEPSEEK_API_KEY environment variable.")
        model = model or DEEPSEEK_MODEL
        url = "https://api.deepseek.com/chat/completions"
        headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        }
        payload = {
            "model": model,
            "messages": [
                {"role": "system", "content": "You are a precise recommendation and user-modelling assistant. Return valid JSON only."},
                {"role": "user", "content": prompt},
            ],
            "temperature": temperature,
            "max_tokens": max_output_tokens,
            "stream": False,
        }
        if json_mode:
            payload["response_format"] = {"type": "json_object"}

        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
        resp.raise_for_status()
        data = resp.json()
        raw_text = data["choices"][0]["message"]["content"]

    else:
        raise ValueError("provider must be 'gemini' or 'deepseek'")

    parsed = None
    if json_mode:
        parsed = safe_json_loads(raw_text)

    return {
        "provider": provider,
        "model": model,
        "raw_text": raw_text,
        "json": parsed,
    }

In [11]:
PROFILE_PROMPT_VERSION = "user_profile_v1"

def make_user_profile_prompt(evidence_packet: Dict[str, Any]) -> str:
    return f"""
You are building a compact behavioural profile for a recommender-system user.

Use ONLY the evidence packet provided below. Do not invent demographics, nationality, age, gender, or life context.
Focus on what will help another model predict:
1. the user's likely star rating for unseen Books or Movies_and_TV items
2. the user's review tone, length, and writing style
3. transferable preferences across Books and Movies_and_TV

Return valid JSON only with this exact structure:
{{
  "user_summary": "2-4 sentence summary of the user's preference pattern",
  "rating_behavior": {{
    "average_rating_interpretation": "...",
    "rating_scale_harshness": "lenient | moderate | harsh | unknown",
    "what_gets_5_stars": "...",
    "what_gets_low_ratings": "...",
    "calibration_rule": "How to anchor future predictions to this user's mean/distribution"
  }},
  "domain_preferences": {{
    "Books": "...",
    "Movies_and_TV": "...",
    "cross_domain": "Domain-agnostic preferences that transfer between books and movies",
    "Genres":"..."
  }},
  "review_style": {{
    "typical_length": "short | medium | long | mixed",
    "tone": "...",
    "specificity": "low | medium | high",
    "common_focus": ["..."],
    "title_style": "...",
    "common_words": "...,..."
  }},
  "useful_prediction_cues": ["..."],
  "uncertainties": ["..."]
}}

Evidence packet:
{json.dumps(evidence_packet, ensure_ascii=False)}
""".strip()

def profile_cache_path(user_id: str, provider: str = LLM_PROVIDER, model: Optional[str] = None) -> Path:
    model = model or (GEMINI_MODEL if provider == "gemini" else DEEPSEEK_MODEL)
    key = f"{PROFILE_PROMPT_VERSION}|{provider}|{model}|{user_id}"
    h = hashlib.sha1(key.encode("utf-8")).hexdigest()
    return PROFILE_CACHE_DIR / f"{h}.json"

def generate_user_profile(
    user_id: str,
    target_domain: Optional[str] = None,
    target_parent_asin: Optional[str] = None,
    provider: str = LLM_PROVIDER,
    model: Optional[str] = None,
    force_refresh: bool = False,
) -> Dict[str, Any]:
    cache_path = profile_cache_path(user_id, provider=provider, model=model)
    if cache_path.exists() and not force_refresh:
        return json.loads(cache_path.read_text())

    evidence_packet = build_user_evidence_packet(
        user_id=user_id,
        target_domain=target_domain,
        target_parent_asin=target_parent_asin,
    )
    prompt = make_user_profile_prompt(evidence_packet)
    result = call_llm(prompt, provider=provider, model=model, json_mode=True, max_output_tokens=1800)

    payload = {
        "user_id": str(user_id),
        "provider": result["provider"],
        "model": result["model"],
        "prompt_version": PROFILE_PROMPT_VERSION,
        "created_at": time.time(),
        "evidence_packet": evidence_packet,
        "profile": result["json"],
    }
    cache_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    return payload

In [47]:
import os
from getpass import getpass

# os.environ["GEMINI_API_KEY"] = getpass("Enter Gemini API key: ")
os.environ["DEEPSEEK_API_KEY"] = getpass("Enter DeepSeek API key: ")
# os.environ["LLM_PROVIDER"] = "deepseek"
# os.environ["LLM_PROVIDER"] = "gemini"

In [37]:
# Test profile generation.


profile_payload = generate_user_profile(
    user_id=sample_row["user_id"],
    provider="gemini",   # or "deepseek",
    model = "gemini-2.5-flash-lite",
    force_refresh=False,
)
print(json.dumps(profile_payload["profile"], indent=2, ensure_ascii=False))

{
  "user_summary": "This user has a strong preference for books, consistently rating them highly with an average of 4.67 stars. They enjoy books with elements of adventure, suspense, romance, and strong characters, though they express a dislike for excessive sex and profanity. Their reviews are typically medium-length, positive in tone, and focus on plot elements and character dynamics.",
  "rating_behavior": {
    "average_rating_interpretation": "The user is a very positive rater, with a high average rating and a strong tendency to give 5-star reviews.",
    "rating_scale_harshness": "lenient",
    "what_gets_5_stars": "Books with adventure, suspense, romance, strong characters, and humor. They appreciate depth in characters and enjoyable plots.",
    "what_gets_low_ratings": "Books that are too heavy or dark, or have an over-abundance of sex and profanity. Length can also be a minor negative factor if it's longer than preferred.",
    "calibration_rule": "Anchor future predictions 

In [17]:
TASK_A_PROMPT_VERSION = "task_a_rating_review_v1"

def make_task_a_prompt(
    user_profile: Dict[str, Any],
    evidence_packet: Dict[str, Any],
    target_item: Dict[str, Any],
    locale_hint: Optional[str] = None,
) -> str:
    locale_instruction = ""
    if locale_hint:
        locale_instruction = f"""
Locale/context instruction:
The user context includes: {locale_hint}
Reflect this only if it is natural for the review style. Avoid caricature, forced slang, or exaggerated pidgin.
""".strip()

    return f"""
You are simulating how a specific Amazon reviewer would rate and review an unseen item.

Use:
- the user's behavioural profile
- the user's selected past reviews
- the target item metadata

Important rules:
- Predict the star rating FIRST, anchored to the user's rating distribution and calibration.
- Then write a review that matches the predicted rating.
- Match the user's usual review length, tone, specificity, and title style.
- Do not mention facts not supported by the item metadata.
- Do not copy any previous review text.
- Return valid JSON only.

Return this exact JSON structure:
{{
  "predicted_rating": 4.0,
  "rating_confidence": "low | medium | high",
  "rating_rationale": "Brief reason, anchored to user profile and item fit.",
  "generated_review_title": "...",
  "generated_review_text": "...",
  "style_match_notes": ["..."],
  "possible_failure_modes": ["..."]
}}

{locale_instruction}

User profile:
{json.dumps(user_profile, ensure_ascii=False)}

Selected past-review evidence:
{json.dumps(evidence_packet["selected_review_examples"], ensure_ascii=False)}

User rating stats:
{json.dumps(evidence_packet["rating_stats"], ensure_ascii=False)}

Target item:
{json.dumps(target_item, ensure_ascii=False)}
""".strip()

def predict_task_a(
    user_id: str,
    target_domain: str,
    target_parent_asin: str,
    provider: str = LLM_PROVIDER,
    model: Optional[str] = None,
    locale_hint: Optional[str] = None,
    force_profile_refresh: bool = False,
) -> Dict[str, Any]:
    evidence_packet = build_user_evidence_packet(
        user_id=user_id,
        target_domain=target_domain,
        target_parent_asin=target_parent_asin,
    )

    profile_payload = generate_user_profile(
        user_id=user_id,
        target_domain=target_domain,
        target_parent_asin=target_parent_asin,
        provider=provider,
        model=model,
        force_refresh=force_profile_refresh,
    )

    target_item = summarize_item(target_domain, target_parent_asin, max_chars=900)
    prompt = make_task_a_prompt(
        user_profile=profile_payload["profile"],
        evidence_packet=evidence_packet,
        target_item=target_item,
        locale_hint=locale_hint,
    )

    result = call_llm(
        prompt,
        provider=provider,
        model=model,
        json_mode=True,
        max_output_tokens=1400,
        temperature=0.25,
    )

    out = result["json"]
    # Basic guardrails.
    try:
        out["predicted_rating"] = max(1.0, min(5.0, float(out["predicted_rating"])))
    except Exception:
        out["predicted_rating"] = None

    return {
        "user_id": str(user_id),
        "target_domain": str(target_domain),
        "target_parent_asin": str(target_parent_asin),
        "provider": result["provider"],
        "model": result["model"],
        "profile_prompt_version": PROFILE_PROMPT_VERSION,
        "task_a_prompt_version": TASK_A_PROMPT_VERSION,
        "target_item": target_item,
        "profile": profile_payload["profile"],
        "prediction": out,
    }

In [39]:
# Single Task A test on one held-out internal validation item.

target = internal_val_df.sample(1, random_state=7).iloc[0]
pred = predict_task_a(
    user_id=target["user_id"],
    target_domain=target["domain"],
    target_parent_asin=target["parent_asin"],
    provider="gemini",  # or "deepseek"
    locale_hint=None,   # try: "Nigerian user; standard Nigerian English, light natural phrasing only"
)
print("GROUND TRUTH RATING:", target["rating"])
print(json.dumps(pred["prediction"], indent=2, ensure_ascii=False))

GROUND TRUTH RATING: 5.0
{
  "predicted_rating": 5.0,
  "rating_confidence": "high",
  "rating_rationale": "The user has a strong preference for action movies and director-driven films, and this item features Roland Emmerich (director) and is an action/sci-fi film. The user's past reviews indicate a high likelihood of 5-star ratings for such content, especially when described as 'great'.",
  "generated_review_title": "Great Action Movie!",
  "generated_review_text": "This is a fantastic action movie with a great storyline. If you like big spectacle and alien invasion films, you'll definitely want to get this set. Roland Emmerich delivers another great film.",
  "style_match_notes": [
    "Review matches the user's typical short length and direct tone.",
    "The title style is similar to previous reviews, using a positive exclamation.",
    "The review mentions genre (action) and director (Roland Emmerich), aligning with user preferences.",
    "Uses common words like 'great' and 'movi

In [43]:
target

domain                                                                                                 Movies_and_TV
user_id                                                                                 AHAMGCZM7YDTTT62Z7BSO5WOBWUQ
asin                                                                                                      B01CZK37C0
parent_asin                                                                                               B01CZK37C0
rating                                                                                                           5.0
review_title                                                                         Lots of action and fun to watch
review_text          Goes along very well with the original Independence Day film.  Lots of action and fun to watch.
timestamp                                                                                              1481903486000
verified_purchase                                               

In [44]:
target_item = summarize_item("Movies_and_TV", "B01CZK37C0", max_chars=900)
print(target_item)

{'domain': 'Movies_and_TV', 'parent_asin': 'B01CZK37C0', 'title': 'INDEPENDENCE DAY', 'store': 'Will Smith (Actor), Bill Pullman (Actor), Roland Emmerich (Director, Writer) &...', 'categories': 'Movies & TV Blu-ray Movies', 'features': '', 'description': "One of the biggest box office hits of all time delivers the ultimate encounter when mysterious and powerful aliens launch an all-out invasion against the human race. The spectacle begins when massic spaceships appear in Earth's skies. But wonder turns to terror as the ships biast destructive beams of fire down on cities all over the planet. Now the world's only hope lies with a determined band of survivors, uniting for one last strike against the invaders - before it's the end of all mankind.", 'price': 9.6}


In [21]:
items_df[items_df['parent_asin'] == 'B0B8JMCCJR']

,domain,parent_asin,main_category,title,store,categories,features,description,price
374488,Movies_and_TV,B0B8JMCCJR,Prime Video,,,,,,NaN


In [48]:
def evaluate_task_a_small_sample(
    n: int = 10,
    provider: str = LLM_PROVIDER,
    random_state: int = 123,
    locale_hint: Optional[str] = None,
    sleep_s: float = 5.0,
) -> pd.DataFrame:
    """
    Very small iteration loop. Use this to compare prompt variants quickly.
    Do not run huge evals through the LLM while iterating.
    """
    rows = internal_val_df.sample(n, random_state=random_state).copy()
    records = []

    for _, row in tqdm(rows.iterrows(), total=len(rows)):
        try:
            pred = predict_task_a(
                user_id=row["user_id"],
                target_domain=row["domain"],
                target_parent_asin=row["parent_asin"],
                provider=provider,
                locale_hint=locale_hint,
            )
            y_true = float(row["rating"])
            y_pred = pred["prediction"].get("predicted_rating")
            err = None if y_pred is None else y_pred - y_true

            records.append({
                "user_id": row["user_id"],
                "domain": row["domain"],
                "parent_asin": row["parent_asin"],
                "true_rating": y_true,
                "pred_rating": y_pred,
                "abs_error": None if err is None else abs(err),
                "sq_error": None if err is None else err ** 2,
                "review_title": row["review_title"],
                "review_text": row["review_text"],
                "pred_review_title": pred["prediction"].get("generated_review_title"), 
                "pred_review_text": pred["prediction"].get("generated_review_text"),
                "pred_rationale": pred["prediction"].get("rating_rationale"),
            })
        except Exception as e:
            records.append({
                "user_id": row["user_id"],
                "domain": row["domain"],
                "parent_asin": row["parent_asin"],
                "true_rating": float(row["rating"]),
                "error": repr(e),
            })

        if sleep_s:
            time.sleep(sleep_s)

    df = pd.DataFrame(records)
    if "sq_error" in df and df["sq_error"].notna().any():
        rmse = math.sqrt(df["sq_error"].dropna().mean())
        mae = df["abs_error"].dropna().mean()
        print(f"n={df['pred_rating'].notna().sum()} RMSE={rmse:.4f} MAE={mae:.4f}")
    return df

# Example:
results_df = evaluate_task_a_small_sample(n=5, provider="deepseek", random_state=3, sleep_s=5.0)
results_df

  0%|          | 0/5 [00:00<?, ?it/s]

n=3 RMSE=0.5774 MAE=0.3333


,user_id,domain,parent_asin,true_rating,pred_rating,abs_error,sq_error,review_title,review_text,pred_review_title,pred_review_text,pred_rationale,error
0,AFHD546QOASX6AORONPD2FXC5CHQ,Books,B07Y8HQ783,5.0,5.0,0.0,0.0,Was surprised at the end.,This series of short stories are so great that you can't put them done till your done reading them. They are filled with compassion and justice.,Another great story,"I couldn't put this down. As always, Dean Koontz delivers a gripping, addictive story that keeps you hooked until the very end. A satisfying conclusion to t...",User previously rated Nameless Series Book 2 with 5 stars and praised its addictive quality; this concluding installment is likely to similarly satisfy thei...,NaN
1,AH66VKOND3WMW4J4KJBY72DEWSCA,Movies_and_TV,B00XR3CKZU,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"JSONDecodeError(""Expecting ',' delimiter: line 23 column 4 (char 3181)"")"
2,AES7PROY4I7EDQUAJER2XDAFRGNQ,Books,B008X3NCRE,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,JSONDecodeError('Invalid control character at: line 21 column 102 (char 2342)')
3,AGL3YAW5MYLT5ORWAI2UUKZ2Q4NA,Books,0385539436,5.0,4.0,1.0,1.0,A great writer at his peak,"I really enjoy just about everything John Grisham writes, though I'm not a huge fan of his short stories(or anyone else's for that matter), and this book fe...",Entertaining,"I've read a few John Grisham novels over the years, and Rogue Lawyer is a solid addition to his lineup. Sebastian Rudd is a refreshingly unconventional prot...","The user enjoys compelling legal thrillers with strong characters and dialogue, as seen in their 4-star review of Mr. Mercedes. John Grisham is a master of ...",NaN
4,AHDCFRQCT62WDRTYDIDK5Q3ZYFIA,Movies_and_TV,B074MJHHLD,5.0,5.0,0.0,0.0,Great Movies,Happy to be able to get all the movies as one set. No issues with these dvds. Happy with the purchase.,Excellent collection!,Great set! Love watching these movies again. So glad to have them all in one package.,"User consistently gives 5 stars to all reviewed items, especially TV series collections. This item matches their preference for complete box sets.",NaN


In [46]:
internal_val_df.head(2)

,domain,user_id,asin,parent_asin,rating,review_title,review_text,timestamp,verified_purchase,helpful_vote,rn,n_user_reviews,split
0,Movies_and_TV,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B0000E2R6P,B0000E2R6P,5.0,EXCELLENT old fashioned Horror,Yea Mr. Silva! EXCELLENT old fashioned Horror!!,1415579707000,True,0,31,40,internal_val
1,Books,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B005OH9A6E,B005OH9A6E,5.0,cool movie!,I haven't finished reading it yet...almost done...I am so gone! Taken away and right in there wit the characters! Dean Koontz has done it again...would make...,1416969499000,True,1,32,40,internal_val


In [ ]:
# Prompt iteration notes:
#
# Baseline to beat on internal validation rating:
#   user_item_bias RMSE ≈ 0.9577
#   user_mean      RMSE ≈ 0.9754
#
# The LLM may not beat these immediately on RMSE.
# Use this notebook to improve:
#   1. rating calibration
#   2. profile compactness
#   3. review tone/style fidelity
#
# Good next experiments:
#   - Add item average_rating/rating_number to target metadata when available.
#   - Add same-domain examples first if Task A target is same-domain.
#   - Add explicit calibration: "user mean is X; predict delta from mean".
#   - Generate rating only first, then review in a second call if rating/review mismatch occurs.